<a href="https://colab.research.google.com/github/minyi-k03/Large-Language-Model-LLM-/blob/Fine-Tuning/PEFT_prompt_tuning_casual_language_modeling_example_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prompt Tuning casual language modeling 예제 - twitter_complaints 데이터셋
## 작성자 : AISchool ([http://aischool.ai/](http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/) )
## Reference : https://huggingface.co/docs/peft/task_guides/clm-prompt-tuning

# 필요한 라이브러리 설치

In [1]:
!pip install -q -U peft transformers accelerate evaluate "datasets==2.21.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


# 설정값 지정

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType, PeftType
import torch
from datasets import load_dataset
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" #NDIIA GPU 사용
model_name_or_path = "bigscience/bloomz-560m" #bloomz-560m 모델 사용
tokenizer_name_or_path = "bigscience/bloomz-560m"

dataset_name = "twitter_complaints" #데이터 셋 설정
text_column = "Tweet text" # 입력 데이터
label_column = "text_label"# 데이터 레이블 분류
max_length = 64 #토큰 max_length
lr = 3e-2 #Learning rate 0.03
num_epochs = 50
batch_size = 8

# twitter_complaints 데이터셋 불러오기
## (RAFT 데이터셋의 부분집합 데이터셋으로 **트위터 문장이 불평(항의)(complaint)인지 아닌지를 분류**하는 데이터셋입니다.)

In [4]:
dataset = load_dataset("ought/raft", dataset_name)
dataset["train"][0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/50 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3399 [00:00<?, ? examples/s]

{'Tweet text': '@HMRCcustomers No this is my first job', 'ID': 0, 'Label': 2}

In [8]:
#Complaint인지 No complaint인지에 대한 분류 작업
classes = [k.replace("_", " ") for k in dataset["train"].features["Label"].names]

dataset = dataset.map(
    lambda x: {"text_label": [classes[label] for label in x["Label"]]},
    batched=True,
    num_proc=1,
)
dataset["train"][0]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/3399 [00:00<?, ? examples/s]

{'Tweet text': '@HMRCcustomers No this is my first job',
 'ID': 0,
 'Label': 2,
 'text_label': 'no complaint'}

In [9]:
dataset["train"][2]

{'Tweet text': "If I can't get my 3rd pair of @beatsbydre powerbeats to work today I'm doneski man. This is a slap in my balls. Your next @Bose @BoseService",
 'ID': 2,
 'Label': 1,
 'text_label': 'complaint'}

# Tokenizer 불러오기 & 데이터셋 전처리

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [11]:
def preprocess_function(examples):
    batch_size = len(examples[text_column])#"Tweet Text"
    inputs = [f"{text_column} : {x} Label : " for x in examples[text_column]] #"Label"
    targets = [str(x) for x in examples[label_column]] #"complaint or no complaint"
    model_inputs = tokenizer(inputs)
    labels = tokenizer(targets)

    #Batch 단위로 묶는다
    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i] + [tokenizer.pad_token_id]
        #print(i, sample_input_ids, label_input_ids)
        model_inputs["input_ids"][i] = sample_input_ids + label_input_ids
        labels["input_ids"][i] = [-100] * len(sample_input_ids) + label_input_ids
        model_inputs["attention_mask"][i] = [1] * len(model_inputs["input_ids"][i])
    #print(model_inputs)


    for i in range(batch_size):
        sample_input_ids = model_inputs["input_ids"][i]
        label_input_ids = labels["input_ids"][i]
        model_inputs["input_ids"][i] = [tokenizer.pad_token_id] * (
            max_length - len(sample_input_ids)
        ) + sample_input_ids
        model_inputs["attention_mask"][i] = [0] * (max_length - len(sample_input_ids)) + model_inputs[
            "attention_mask"
        ][i]
        labels["input_ids"][i] = [-100] * (max_length - len(sample_input_ids)) + label_input_ids
        model_inputs["input_ids"][i] = torch.tensor(model_inputs["input_ids"][i][:max_length])
        model_inputs["attention_mask"][i] = torch.tensor(model_inputs["attention_mask"][i][:max_length])
        labels["input_ids"][i] = torch.tensor(labels["input_ids"][i][:max_length])

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [12]:
processed_datasets = dataset.map(
    preprocess_function,
    batched=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names,
    load_from_cache_file=False,
    desc="Running tokenizer on dataset",
)

Running tokenizer on dataset:   0%|          | 0/50 [00:00<?, ? examples/s]

Running tokenizer on dataset:   0%|          | 0/3399 [00:00<?, ? examples/s]

In [13]:
train_dataset = processed_datasets["train"]
eval_dataset = processed_datasets["test"]

train_dataloader = DataLoader(
    train_dataset, shuffle=True, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True
)
eval_dataloader = DataLoader(eval_dataset, collate_fn=default_data_collator, batch_size=batch_size, pin_memory=True)

# PEFT 모델 설정

In [16]:
# PEFT를 이용한 PromtTuning 설정
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.TEXT,
    num_virtual_tokens=8, #Embedding Token 개수
    prompt_tuning_init_text="Classify if the tweet is a complaint or not:", #정보값을 가진 문자을 초기 토큰으로 부여
    tokenizer_name_or_path=model_name_or_path,
)

# Prompt Tuning 기법으로 인해 전체 모델의 0.0014%의 파라미터만 Fine-Tuning에 사용

In [17]:
#Prompt-tuning 적용
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
#5억 5천만개 파라미터에서 8,192파라미터 개수를 줄임

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

trainable params: 8,192 || all params: 559,222,784 || trainable%: 0.0015


# Training 시작

In [18]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
lr_scheduler = get_linear_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=(len(train_dataloader) * num_epochs),
)

In [19]:
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, batch in enumerate(tqdm(train_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_loss += loss.detach().float()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

    model.eval()
    eval_loss = 0
    eval_preds = []
    for step, batch in enumerate(tqdm(eval_dataloader)):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)
        loss = outputs.loss
        eval_loss += loss.detach().float()
        eval_preds.extend(
            tokenizer.batch_decode(torch.argmax(outputs.logits, -1).detach().cpu().numpy(), skip_special_tokens=True)
        )

    eval_epoch_loss = eval_loss / len(eval_dataloader)
    eval_ppl = torch.exp(eval_epoch_loss)
    train_epoch_loss = total_loss / len(train_dataloader)
    train_ppl = torch.exp(train_epoch_loss)
    print(f"{epoch=}: {train_ppl=} {train_epoch_loss=} {eval_ppl=} {eval_epoch_loss=}")

100%|██████████| 425/425 [01:31<00:00,  4.63it/s]


epoch=0: train_ppl=tensor(8.2388e+16, device='cuda:0') train_epoch_loss=tensor(38.9502, device='cuda:0') eval_ppl=tensor(16207.7070, device='cuda:0') eval_epoch_loss=tensor(9.6932, device='cuda:0')


100%|██████████| 425/425 [01:40<00:00,  4.23it/s]


epoch=1: train_ppl=tensor(750887.9375, device='cuda:0') train_epoch_loss=tensor(13.5290, device='cuda:0') eval_ppl=tensor(9100.5908, device='cuda:0') eval_epoch_loss=tensor(9.1161, device='cuda:0')


100%|██████████| 425/425 [01:40<00:00,  4.25it/s]


...

epoch=47: train_ppl=tensor(1.5832, device='cuda:0') train_epoch_loss=tensor(0.4595, device='cuda:0') eval_ppl=tensor(76190.6094, device='cuda:0') eval_epoch_loss=tensor(11.2410, device='cuda:0')


100%|██████████| 425/425 [01:39<00:00,  4.26it/s]


epoch=48: train_ppl=tensor(1.5674, device='cuda:0') train_epoch_loss=tensor(0.4494, device='cuda:0') eval_ppl=tensor(77740.9922, device='cuda:0') eval_epoch_loss=tensor(11.2611, device='cuda:0')


100%|██████████| 425/425 [01:39<00:00,  4.26it/s]

epoch=49: train_ppl=tensor(1.5569, device='cuda:0') train_epoch_loss=tensor(0.4427, device='cuda:0') eval_ppl=tensor(80280., device='cuda:0') eval_epoch_loss=tensor(11.2933, device='cuda:0')


# 학습이 끝난 모델을 sample text에 대한 Inference

In [20]:
inputs = tokenizer(
    f'{text_column} : {"@nationalgridus I have no water and the bill is current and paid. Can you do something about this?"} Label : ',
    return_tensors="pt",
)

In [21]:
model.to(device)

with torch.no_grad():
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(
        input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], max_new_tokens=10, eos_token_id=3
    )
    print(tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True))

['Tweet text : @nationalgridus I have no water and the bill is current and paid. Can you do something about this? Label : complaint']
